# AIM 402 - My Course Dataset (simulated)

Simulated dataset: **Nepal Student Performance** (1,200 students).
Generated with NumPy using a fixed seed, so the same data is created every time.
Some values are missing on purpose, to make it behave like real survey data.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(402)   # fixed seed = same dataset every run
n = 1200

# ---------- NUMERIC VARIABLES ----------
age = rng.integers(17, 25, n)                                        # 17-24
study_hours = np.clip(rng.gamma(2.5, 1.6, n), 0, 14)                 # per day, right-skewed
sleep_hours = np.clip(rng.normal(6.8, 1.2, n), 3, 10)                # per night
attendance_pct = np.clip(rng.normal(82, 11, n), 30, 100)             # %
family_income = np.round(rng.lognormal(3.6, 0.6, n), 1)              # NPR thousand / month, skewed

# ---------- CATEGORICAL VARIABLES ----------
location_type = rng.choice(["Urban", "Semi-urban", "Rural"], n, p=[0.45, 0.30, 0.25])        # 3 levels
stream = rng.choice(["Science", "Management", "Humanities", "Education"], n,
                    p=[0.30, 0.35, 0.20, 0.15])                                              # 4 levels

# binary: tuition more likely when income is higher
p_tuition = 1 / (1 + np.exp(-(np.log(family_income) - 3.6) * 1.5))
has_tuition = np.where(rng.random(n) < p_tuition, "Yes", "No")                               # binary

# ---------- EXAM SCORE (depends on the other variables + noise) ----------
loc_effect = np.select([location_type == "Urban", location_type == "Semi-urban"], [3, 0], default=-3)
stream_effect = np.select([stream == "Science", stream == "Management", stream == "Humanities"],
                          [2, 1, -1], default=0)
noise = rng.normal(0, 7, n)

exam_score = (12 + 4.0 * study_hours + 0.25 * attendance_pct + 1.5 * (sleep_hours - 6.8)
              + 6 * (has_tuition == "Yes") + loc_effect + stream_effect + noise)
exam_score = np.round(np.clip(exam_score, 0, 100), 1)

result = np.where(exam_score >= 40, "Pass", "Fail")                                          # binary

# ---------- PUT IN A TABLE ----------
df = pd.DataFrame({
    "student_id": np.arange(1, n + 1),
    "age": age,
    "study_hours": np.round(study_hours, 1),
    "sleep_hours": np.round(sleep_hours, 1),
    "attendance_pct": np.round(attendance_pct, 1),
    "family_income": family_income,
    "location_type": location_type,
    "stream": stream,
    "has_tuition": has_tuition,
    "exam_score": exam_score,
    "result": result,
})

# ---------- ADD MISSING VALUES (random blanks) ----------
missing_rate = {"study_hours": 0.03, "sleep_hours": 0.05, "attendance_pct": 0.04,
                "family_income": 0.12, "location_type": 0.02, "has_tuition": 0.03}

for col, rate in missing_rate.items():
    mask = rng.random(n) < rate
    df[col] = df[col].where(~mask, np.nan)

# a few unusual (outlier) values, like real data entry mistakes
bad = rng.choice(n, 6, replace=False)
df.loc[bad[:3], "study_hours"] = 16.0      # unrealistic study hours
df.loc[bad[3:], "family_income"] = 950.0   # very high income

df.to_csv("nepal_student_performance.csv", index=False)
print("Saved:", df.shape)
df.head()

Saved: (1200, 11)


,student_id,age,study_hours,sleep_hours,attendance_pct,family_income,location_type,stream,has_tuition,exam_score,result
0,1,22,2.6,7.1,85.6,18.6,Rural,Humanities,No,42.3,Pass
1,2,19,3.9,8.0,61.0,32.9,Urban,Management,Yes,60.3,Pass
2,3,18,3.3,8.1,77.7,54.8,Semi-urban,Humanities,Yes,33.6,Fail
3,4,23,7.2,7.5,62.2,61.1,Semi-urban,Science,Yes,68.6,Pass
4,5,24,1.6,8.1,75.1,NaN,NaN,Education,Yes,50.3,Pass


## Quick checks

In [2]:
print(df.shape)
print()
print("Missing values per column:")
print(df.isna().sum())
print()
df.describe()

(1200, 11)

Missing values per column:
student_id          0
age                 0
study_hours        38
sleep_hours        72
attendance_pct     51
family_income     133
location_type      30
stream              0
has_tuition        33
exam_score          0
result              0
dtype: int64



,student_id,age,study_hours,sleep_hours,attendance_pct,family_income,exam_score
count,1200.000000,1200.000000,1162.000000,1128.000000,1149.000000,1067.000000,1200.000000
mean,600.500000,20.571667,4.013167,6.814362,81.511314,46.815651,53.124083
std,346.554469,2.272480,2.658208,1.189121,10.648389,56.059681,13.728539
min,1.000000,17.000000,0.200000,3.000000,48.900000,4.200000,18.700000
25%,300.750000,19.000000,2.100000,6.000000,74.100000,24.500000,43.875000
50%,600.500000,21.000000,3.400000,6.800000,81.800000,37.100000,51.500000
75%,900.250000,23.000000,5.300000,7.600000,89.300000,55.400000,61.500000
max,1200.000000,24.000000,16.000000,10.000000,100.000000,950.000000,100.000000


In [3]:
# Does it meet the course requirements?
num_cols = df.select_dtypes("number").columns.drop("student_id").tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

print("Rows >= 500        :", len(df) >= 500)
print("Numeric columns    :", num_cols)
print("Categorical columns:", {c: df[c].nunique() for c in cat_cols})
print("Has a binary column       :", any(df[c].nunique() == 2 for c in cat_cols))
print("Has a 3+ level column     :", any(df[c].nunique() >= 3 for c in cat_cols))

Rows >= 500        : True
Numeric columns    : ['age', 'study_hours', 'sleep_hours', 'attendance_pct', 'family_income', 'exam_score']
Categorical columns: {'location_type': 3, 'stream': 4, 'has_tuition': 2, 'result': 2}
Has a binary column       : True
Has a 3+ level column     : True


In [4]:
# ================= CLEANING PART =================

# 1. keep the raw (with missing values) as a backup
df.to_csv("nepal_student_performance_raw.csv", index=False)

# 2. fill numeric columns with the MEDIAN
df_clean = df.copy()
for col in ["study_hours", "sleep_hours", "attendance_pct", "family_income"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 3. fill categorical columns with the MOST COMMON value
for col in ["location_type", "has_tuition"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# 4. save the clean data (same file name, now with NO missing values)
df_clean.to_csv("nepal_student_performance.csv", index=False)

# 5. check
print("Missing values left:", df_clean.isna().sum().sum())   # 0
print("Shape:", df_clean.shape)                               # (1200, 11)
print(pd.read_csv("nepal_student_performance.csv").isna().sum().sum())  # 0 in the saved file

Missing values left: 0
Shape: (1200, 11)
0
